# Downloading Data
1. Load StatsBomb open data in Jupyter with mplsoccer's Sbopen, and list which competitions include 360 freeze frames
2. Download Metrica Sports' sample tracking matches and SkillCorner's open data

## StatsBomb Data

In [1]:
#importing SBopen class from mplsoccer to open the data
from mplsoccer import Sbopen
import pandas as pd
# The first thing we have to do is open the data. We use a parser SBopen available in mplsoccer.
parser = Sbopen()

##############################################################################
# Competition data
# ----------------------------
# Using method *competition* of the parser we can explore competitions to find the competition we are interested in.
# The most important information for us is in the *competition_id* (id of competition) and *season_id*.
# The first one is the key in Statsbomb database of a competition, the second one of a season 
# of this competition (for example WC 2018 would have a different *season_id* than WC 2014, but the same *competition_id*).

#opening data using competition method
df_competition = parser.competition()
#structure of data
df_competition.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   competition_id             80 non-null     int64
 1   season_id                  80 non-null     int64
 2   country_name               80 non-null     str  
 3   competition_name           80 non-null     str  
 4   competition_gender         80 non-null     str  
 5   competition_youth          80 non-null     bool 
 6   competition_international  80 non-null     bool 
 7   season_name                80 non-null     str  
 8   match_updated              80 non-null     str  
 9   match_updated_360          58 non-null     str  
 10  match_available_360        12 non-null     str  
 11  match_available            80 non-null     str  
dtypes: bool(2), int64(2), str(8)
memory usage: 14.8 KB


In [2]:
# Find rows where match_available_360 is non-null
rows_with_360 = df_competition[df_competition['match_available_360'].notna()]

# Display the rows
rows_with_360[['competition_id', 'season_id', 'competition_name', 'season_name']]

,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
30,43,106,FIFA World Cup,2022
40,11,90,La Liga,2020/2021
61,7,235,Ligue 1,2022/2023
62,7,108,Ligue 1,2021/2022
64,44,107,Major League Soccer,2023
73,55,282,UEFA Euro,2024
74,55,43,UEFA Euro,2020
76,53,315,UEFA Women's Euro,2025


In [3]:
# Alternatively, to just see the index of these rows:
print("\nIndex of rows with non-null match_available_360:")
print(rows_with_360.index.tolist())


Index of rows with non-null match_available_360:
[0, 2, 30, 40, 61, 62, 64, 73, 74, 76, 77, 78]


In [4]:
# determine how many games in each competition/season contain 360 data
results = []
for row in rows_with_360.itertuples():
    df_match = parser.match(competition_id=row.competition_id, season_id=row.season_id)
    results.append({'competition_id': row.competition_id,
                    'season_id': row.season_id,
                    'competition': row.competition_name,
                    'season': row.season_name,
                    'total games': len(df_match),
                    '360 games': (df_match['match_status_360'] == 'available').sum(),
                   })
#save table as CSV file for later reference/use
df_360 = pd.DataFrame(results)
df_360.to_csv('data/statsbomb_360_coverage.csv', index=False)
df_360

,competition_id,season_id,competition,season,total games,360 games
0,9,281,1. Bundesliga,2023/2024,34,34
1,1267,107,African Cup of Nations,2023,52,1
2,43,106,FIFA World Cup,2022,64,64
3,11,90,La Liga,2020/2021,35,35
4,7,235,Ligue 1,2022/2023,32,32
5,7,108,Ligue 1,2021/2022,26,26
6,44,107,Major League Soccer,2023,6,6
7,55,282,UEFA Euro,2024,51,51
8,55,43,UEFA Euro,2020,51,51
9,53,315,UEFA Women's Euro,2025,31,31


In [5]:
# list competitions with full data (from dictionary defined in soccer_utils.py) as competition_id, season_id
from soccer_utils import MENS_TOURNAMENTS_360
MENS_TOURNAMENTS_360

{'World Cup 2022': (43, 106), 'Euro 2020': (55, 43), 'Euro 2024': (55, 282)}

In [6]:
# testing load_events() from soccer_utils.py with small data set (2023 MLS)

from soccer_utils import load_events
events = load_events(44, 107)
events.shape

Loaded events from cache


(21786, 81)

In [7]:
#testing load_many() from soccer_utils.py with Women's data set
import soccer_utils as su

events = su.load_many(su.WOMENS_TOURNAMENTS_360)
summary = events.groupby('tournament').agg(
    matches=('match_id', 'nunique'),
    events=('id', 'count'),
)
summary['events_per_match'] = (summary['events'] / summary['matches']).round()
summary

Loaded events from cache
Loaded events from cache
Loaded events from cache


,matches,events,events_per_match
tournament,,,
Women's Euro 2022,31,105141,3392.0
Women's Euro 2025,31,105658,3408.0
Women's World Cup 2023,64,226118,3533.0


In [8]:
events.head()

,id,index,period,timestamp,minute,second,possession,duration,match_id,type_id,...,block_save_block,ball_recovery_offensive,pass_no_touch,pass_miscommunication,dribble_no_touch,shot_redirect,player_off_permanent,shot_follows_dribble,tournament,half_start_late_video_start
0,248b79eb-0bea-488e-a037-a5f8fc4ef6d0,1,1,00:00:00,0,0,1,0.000000,3893834,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
1,91acd412-face-4384-8bea-1da7f5d0a691,2,1,00:00:00,0,0,1,0.000000,3893834,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
2,f108aa25-68d2-46ab-9d01-5e90cc24b801,3,1,00:00:00,0,0,1,0.000000,3893834,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
3,e272e6e0-e5be-4fd8-ae5f-a357d3f304e9,4,1,00:00:00,0,0,1,0.000000,3893834,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
4,4ab6530b-5b79-4c23-8f80-095cc8b7b722,5,1,00:00:00.694000,0,0,2,1.367021,3893834,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN


In [9]:
# load men's tournaments (from cache after the first run) and summarise
events = su.load_many(su.MENS_TOURNAMENTS_360)

summary = events.groupby('tournament').agg(
    matches=('match_id', 'nunique'),
    events=('id', 'count'),
)
summary['events_per_match'] = (summary['events'] / summary['matches']).round()
summary

Loaded events from cache
Loaded events from cache
Loaded events from cache


,matches,events,events_per_match
tournament,,,
Euro 2020,51,192664,3778.0
Euro 2024,51,187924,3685.0
World Cup 2022,64,234637,3666.0


In [10]:
events.head()

,id,index,period,timestamp,minute,second,possession,duration,match_id,type_id,...,ball_recovery_offensive,pass_no_touch,shot_redirect,bad_behaviour_card_id,bad_behaviour_card_name,block_save_block,shot_follows_dribble,half_start_late_video_start,tournament,player_off_permanent
0,1f60fe1e-0340-49e2-975e-f16f54c5da1a,1,1,00:00:00,0,0,1,0.00000,3857276,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
1,d5ffdab0-a19a-497e-a0f6-d5403bfcc2b5,2,1,00:00:00,0,0,1,0.00000,3857276,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
2,c6dcab53-5a07-4a6e-ab8b-64e9f9133d19,3,1,00:00:00,0,0,1,0.00000,3857276,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
3,396c6ebb-5ba7-415c-ba37-88ff51f7c737,4,1,00:00:00,0,0,1,0.00000,3857276,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
4,05a57281-9295-408a-9c4f-e32d7fad4d96,5,1,00:00:00.460000,0,0,2,0.83469,3857276,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN


## Metrica Sports